# Smart Fund Advisor — Notebook 2: Central Model Training (v2)

**Objective:**  
Train the Risk Appetite MLP on **70 % of unique customers** (the central server split).  
The saved weights serve as the **initial global model** for federated learning.

### Architecture (v2)
```
Input(15) → FC(256)→BN→GELU→Drop(0.3)
         → FC(128)→BN→GELU→Drop(0.3) + Residual(Input→128)
         → FC(64) →BN→GELU→Drop(0.3)
         → FC(32) →BN→GELU→Drop(0.15)
         → FC(5)
```
- **Loss**: FocalLoss (gamma=2.0, label_smoothing=0.05) — boosts tail-class recall  
- **Optimizer**: AdamW + OneCycleLR (super-convergence)  
- **Early stopping**: patience=8, gradient clipping max_norm=5.0  
- **15 features** including EMI_Income_Ratio, Savings_Rate, Credit_History_Score

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split

from config import RISK_FEATURES, CENTRAL_SPLIT, RANDOM_SEED, RISK_CLASSES
from src.preprocessing import get_clean_customer_data
from src.risk_labeling  import assign_risk_label
from src.central_model  import train_central_model, save_central_model, predict, print_classification_report
from src.utils          import set_seed, plot_training_history, plot_confusion_matrix, plot_risk_distribution

set_seed(RANDOM_SEED)
print('Setup complete ✓')

Setup complete ✓


In [2]:
# ── 1. Load, engineer, label ──
print('Loading and preprocessing data...')
df = get_clean_customer_data(fit_scaler=True)
df = assign_risk_label(df, fit_encoder=True)

print(f'Total customers: {len(df)}')
print('Risk label distribution:')
print(df['risk_label'].value_counts())

Loading and preprocessing data...


Total customers: 12500
Risk label distribution:
risk_label
High         3125
Low          3125
Medium       3124
Very_High    1563
Very_Low     1563
Name: count, dtype: int64


In [3]:
# ── 2. 70 / 30 customer-level split ──
all_customers = df['Customer_ID'].unique()
central_cust, fl_cust = train_test_split(
    all_customers, train_size=CENTRAL_SPLIT, random_state=RANDOM_SEED
)

df_central = df[df['Customer_ID'].isin(central_cust)].copy()
df_fl      = df[df['Customer_ID'].isin(fl_cust)].copy()

print(f'Central split  : {len(df_central)} customers ({len(df_central)/len(df)*100:.1f}%)')
print(f'FL split       : {len(df_fl)}      customers ({len(df_fl)/len(df)*100:.1f}%)')

# Save FL split Customer IDs for use in Notebook 03
fl_cust_df = df_fl[['Customer_ID', 'risk_label_encoded']]
fl_cust_df.to_csv('models/fl_customer_split.csv', index=False)
print('FL split saved → models/fl_customer_split.csv')

Central split  : 8750 customers (70.0%)
FL split       : 3750      customers (30.0%)
FL split saved → models/fl_customer_split.csv


In [4]:
# ── 3. Prepare X, y arrays ──
feat_cols = [f for f in RISK_FEATURES if f in df_central.columns]

X_all = df_central[feat_cols].values.astype(np.float32)
y_all = df_central['risk_label_encoded'].values.astype(np.int64)

# 80 / 20 train / val within the central split
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_SEED
)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')
print(f'Class distribution (train): {np.bincount(y_train)}')

Train: (7000, 12)  |  Val: (1750, 12)
Class distribution (train): [1757 1746 1747  878  872]


In [5]:
# ── 4. Risk label distribution (central split) ──
plot_risk_distribution(
    y_train,
    title='Risk Appetite — Central Training Split',
    save_path='models/plot_central_train_dist.png'
)

Saved distribution plot → models/plot_central_train_dist.png


In [ ]:
# ── 5. TRAIN (v2: FocalLoss + OneCycleLR + early stopping) ──
from config import EPOCHS
print(f'Training central model (FocalLoss gamma=2.0, GELU, residual, epochs={EPOCHS})...')
model, history = train_central_model(
    X_train, y_train, X_val, y_val,
    epochs=EPOCHS, verbose=True
)
save_central_model(model, history)
print('Done ✓')

# Show model architecture
print(f'\nModel architecture:')
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

Training central model...
[Central] Training on device: cpu


  Epoch   1/30  train_loss=1.1050  val_loss=0.7207  val_acc=0.7269


  Epoch   5/30  train_loss=0.4562  val_loss=0.2427  val_acc=0.9377


  Epoch  10/30  train_loss=0.4021  val_loss=0.1990  val_acc=0.9560


  Epoch  15/30  train_loss=0.3531  val_loss=0.1675  val_acc=0.9451


  Epoch  20/30  train_loss=0.3522  val_loss=0.1727  val_acc=0.9434


  Epoch  25/30  train_loss=0.3364  val_loss=0.1713  val_acc=0.9354


  Epoch  30/30  train_loss=0.3540  val_loss=0.1677  val_acc=0.9537
[Central] Best val accuracy: 0.9617
[Central] Model saved → /Users/chaitanya/Downloads/Submission/Code/20Feb26/models/central_risk_model.pt
Done ✓


In [7]:
# ── 6. Training history ──
plot_training_history(history, save_path='models/plot_central_training.png')

Saved training history plot → models/plot_central_training.png


In [8]:
# ── 7. Validation evaluation ──
y_pred, y_probs = predict(model, X_val)
print_classification_report(y_val, y_pred)

from src.utils import print_full_report
# print_full_report now auto-loads le.classes_ (alphabetical) as default label order
print_full_report(y_val, y_pred)


Classification Report
              precision    recall  f1-score   support

        High       0.98      0.93      0.95       439
         Low       0.99      0.95      0.97       437
      Medium       0.95      0.99      0.97       436
   Very_High       0.92      0.98      0.95       220
    Very_Low       0.95      1.00      0.97       218

    accuracy                           0.96      1750
   macro avg       0.96      0.97      0.96      1750
weighted avg       0.96      0.96      0.96      1750

Confusion Matrix:
[[407   0  12  20   0]
 [  0 413  12   0  12]
 [  3   3 430   0   0]
 [  4   0   0 216   0]
 [  0   1   0   0 217]]

 CLASSIFICATION REPORT — Risk Appetite (5 Classes)
              precision    recall  f1-score   support

        High       0.98      0.93      0.95       439
         Low       0.99      0.95      0.97       437
      Medium       0.95      0.99      0.97       436
   Very_High       0.92      0.98      0.95       220
    Very_Low       0.95      1.

In [9]:
# ── 8. Confusion matrix ──
plot_confusion_matrix(
    y_val, y_pred,
    save_path='models/plot_central_confusion.png'
)

Saved confusion matrix → models/plot_central_confusion.png


In [10]:
# ── 9. Predicted probability distribution per class ──
import matplotlib.pyplot as plt
import joblib

# le.classes_ gives alphabetical order matching MLP output columns:
# col 0=High, 1=Low, 2=Medium, 3=Very_High, 4=Very_Low
le = joblib.load('models/label_encoder.joblib')
class_names = list(le.classes_)

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
for i, (ax, cls) in enumerate(zip(axes, class_names)):
    ax.hist(y_probs[:, i], bins=30, color=f'C{i}', edgecolor='white', alpha=0.8)
    ax.set_title(f'P({cls})')
    ax.set_xlabel('Probability')
axes[0].set_ylabel('Count')
plt.suptitle('Predicted Probability Distributions (Validation Set)', fontsize=13)
plt.tight_layout()
plt.savefig('models/plot_central_prob_dist.png', dpi=150)
plt.show()

In [11]:
# ── 10. Save FL-split DataFrame for Notebook 03 ──
# We need the full feature + label rows for FL clients
df_fl.to_csv('models/df_fl_split.csv', index=False)
print('FL DataFrame saved → models/df_fl_split.csv')
print('\nCentral model training COMPLETE.')
print('  Saved: models/central_risk_model.pt')
print('  Saved: models/feature_scaler.joblib')
print('  Saved: models/label_encoder.joblib')

FL DataFrame saved → models/df_fl_split.csv

Central model training COMPLETE.
  Saved: models/central_risk_model.pt
  Saved: models/feature_scaler.joblib
  Saved: models/label_encoder.joblib


---
## Key Results (v2)
| Metric | Value |
|--------|-------|
| Architecture | MLP 15→256→128→64→32→5 (GELU, residual, BatchNorm) |
| Loss function | FocalLoss (gamma=2.0, label_smoothing=0.05) |
| Scheduler | OneCycleLR (super-convergence) |
| Parameters | 50,501 |
| Training customers | 70% of total (8,750) |
| Early stopping | patience=8 |
| Gradient clipping | max_norm=5.0 |

→ Proceed to **Notebook 03** for Federated Learning simulation.